# variableMeanDriftingGrating

Protocol-specific analysis. `demos/meaAnalysisMain.ipynb` is the shared front half — it finds datasets and builds a pipeline, then stops before conditions get interpreted, because that is where protocols stop resembling each other. This picks up there, and runs standalone: set the constants in §1 from that notebook's §6 printout and run from the top. A `pipeline` already in the kernel for the same datafile is reused rather than rebuilt.

**The protocol**: a sinewave grating drifts at a fixed temporal frequency while two things alternate across epochs — background **mean intensity** and **bar width**. Every epoch is one cell of that mean × bar-width grid.

Two rules the section order encodes:

- **Epochs first, then cells** (§3 before §4). The other order scores every cell against a stretch of block you were going to discard, which reports a property of the block as a property of each cell.
- **Every stimulus value comes from the recorded epochs**, never from the `.m` defaults.

## 1. Setup and the dataset

- `PROTOCOL_NAME` is the full dotted name as the database stores it, not the search fragment — §2 uses it to find the MATLAB source.
- `ANALYSIS_CHUNK = None` picks the nearest noise chunk with a typing file, the same rule the main notebook uses.
- Entry 4   20230313C	data018	chunk2	
- Entry 5	20230502C	data017	chunk2	
- Entry 6	20230523C	data014	chunk3	

In [1]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# <-- EDIT ME: from the main notebook's §6 printout.
EXP_NAME       = '20230313C'
DATAFILE_NAME  = 'data018'
ANALYSIS_CHUNK = 'chunk2'     # None to let the pipeline pick

PROTOCOL_NAME = 'edu.washington.riekelab.chris.protocols.variableMeanDriftingGrating'
MAIN_TYPES    = ['OnP', 'OffP', 'OnM', 'OffM']

# Reuse a pipeline from the same kernel when it is already the right dataset;
# otherwise build one. Makes the notebook runnable standalone without
# rebuilding needlessly when it isn't.
_existing = globals().get('pipeline')
if (_existing is not None
        and getattr(_existing.resp, 'datafile_name', None) == DATAFILE_NAME
        and _existing.analysis_chunk.exp_name == EXP_NAME):
    print(f'Reusing the pipeline already in this kernel: {EXP_NAME}/{DATAFILE_NAME}')
else:
    pipeline = ra.create_mea_pipeline(EXP_NAME, DATAFILE_NAME,
                                      analysis_chunk_name = ANALYSIS_CHUNK)

stim_block     = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk

Initializing StimBlock for 20230313C block 1610
For Rig C 20230313C:
{'disp_type': 'OLED', 'mu_per_pixel': 3.8, 'n_ht': 600, 'n_wt': 800, 'mean_frame_rate': 60.31807657, 'stage_frame_rate': np.float64(60.0), 'mea_rotation_deg': 90.0}
Using user-specified noise chunk for data018: chunk2.

Initializing ResponseBlock for 20230313C block 1610
If this is for an LED stimulus, be sure to set b_LED=True!

Error occurred while getting actual onset/offset times: unsupported operand type(s) for *: 'float' and 'NoneType'
It could be that frame_times_ms do not have the correct number of frames due to some error in frame detection.
Check the frame monitor sample rate! On MEA Rigs, prefer 1k, errors likely with 10k.
Loading VCD from /Volumes/data/data/sorted/20230313C/data018/kilosort2 ...
VCD loaded with 1452 cells.

Loading VCD from /Volumes/data/analysis/20230313C/chunk2/kilosort2 ...
VCD loaded with 1494 cells.


Loaded spatial maps for channels [0, 2] and 1487 cells of shape (75, 100, 2)
Spatial

### Merge duplicate clusters before cell cleanup

Kilosort can split one physical cell into clusters with near-identical electrical images (EIs). `ra.dedup_pipeline` finds connected groups of those clusters, keeps the typed member with the strongest EI, and unions the spike trains with a short refractory window. This mutates `pipeline.resp` before the epoch summary and silent-cell QC below, so every downstream population analysis counts the merged cell once.

In [2]:
# Protocol-agnostic duplicate-cluster merge; safe to rerun on this pipeline.
DEDUP_EI_THRESHOLD  = 0.85
DEDUP_REFRACTORY_MS = 0.5

_n_cells_before = len(response_block.df_spike_times)
dedup_log = ra.dedup_pipeline(
    pipeline,
    ei_threshold=DEDUP_EI_THRESHOLD,
    merge_strategy='union',
    refractory_ms=DEDUP_REFRACTORY_MS,
    skip_untyped=True,
    verbose=True,
)
_n_cells_after = len(response_block.df_spike_times)
print(f'Protocol cells: {_n_cells_before} -> {_n_cells_after} '
      f'({_n_cells_before - _n_cells_after} duplicates merged)')

if not dedup_log['protocol'].empty:
    display(dedup_log['protocol'])

Deduplicating protocol side (protocol)…
find_duplicate_groups: 4 group(s), 8 cells affected, 4 would be dropped as duplicates (EI≥0.85, same-type)
apply_dedup: kept 4 representative(s), dropped 4 duplicate cell(s); recovered 909 spikes via union (0.5 ms refractory)
Protocol cells: 1452 -> 1448 (4 duplicates merged)


,group,representative,dropped,representative_amp,cell_type,n_spikes_rep_before,n_spikes_dropped_total,n_spikes_rep_after,n_spikes_added_to_rep
0,"(712, 1555)",712,"(1555,)",499.929779,OffP,2500,453,2538,38
1,"(816, 818)",816,"(818,)",419.990234,OffP,2856,1080,3031,175
2,"(909, 1586)",909,"(1586,)",303.674255,OffP,2550,1143,2552,2
3,"(1216, 1734)",1216,"(1734,)",149.428635,OffP,2070,700,2764,694


## 2. What the block ran

- **The epochs are the record; the `.m` declares defaults.** A default is what a parameter would have been had nobody touched it — this protocol declares 4 Hz and an 800 µm aperture, and the block below ran 2 Hz and 2000. A condition axis is likewise one that varies *in the data*, not one declared to vary.
- **The source supplies the one thing the data cannot**: the comment saying what each parameter *is*. `ra.parse_protocol_source` maps the dotted name onto MATLAB's package layout to find the `.m` in the cloned [chris-package](https://github.com/Rieke-Lab/chris-package/) and prints the GitHub URL. Skip it and the tables come out the same, minus the descriptions.
- **Two tables.** *Settings* — the protocol's own parameters that held still for the block (`declared_only=False` adds the 15 the rig recorded: `NDF`, `micronsPerPixel`, `monitorRefreshRate`). *Per epoch* — one row each, the parameters that **changed**, beside that epoch's population spike count, which is what makes §3's range choosable by eye.
- **This protocol misspells its own epoch parameter**: `currentBarWdith`, so anything reaching for `currentBarWidth` silently gets nothing. `ra.condition_keys` surfaces mismatches like that by reporting declared per-epoch names that never vary. Inherited parameters count as undeclared (`preTime`, `tailTime`, from `RiekeLabStageProtocol`) and live in `stim_block.d_epoch_block_params`, where §4 reads epoch length from.

In [3]:
source = ra.parse_protocol_source(PROTOCOL_NAME)

if source is not None:
    print(f'{source.class_name}  <  {source.superclass}')
    print(f'github : {source.github_url}\n')

params = ra.block_parameters(stim_block, source = source)

# Fixed for the whole block — the protocol's own configuration.
print(f"{len(params)} parameters declared by {source.class_name if source else 'the protocol'}; "
      f"{params.attrs.get('n_undeclared', 0)} more the rig recorded are left out "
      f"(declared_only=False to see them).\n")
print(f'Settings held constant across all {len(stim_block.df_epochs)} epochs:')
display(params.query('not epoch_specific')[['parameter', 'value', 'comment']]
              .reset_index(drop = True))

# Varying — the condition axes, with the levels each one took.
print('\nParameters that change from epoch to epoch (the condition axes):')
display(params.query('epoch_specific')[['parameter', 'value', 'n_levels', 'comment']]
              .reset_index(drop = True))

CONDITION_KEYS = ra.condition_keys(stim_block, source = source)

variableMeanDriftingGrating  <  edu.washington.riekelab.protocols.RiekeLabStageProtocol
github : https://github.com/Rieke-Lab/chris-package/blob/master/+edu/+washington/+riekelab/+chris/+protocols/variableMeanDriftingGrating.m

13 parameters declared by variableMeanDriftingGrating; 15 more the rig recorded are left out (declared_only=False to see them).

Settings held constant across all 20 epochs:


,parameter,value,comment
0,amp,Amp1,Output amplifier
1,apertureDiameter,2000.0,Surround radius (pix)
2,barWidths,"[50.0, 150.0]",Center bar width (pix)
3,meanIntensities,"[0.03, 0.3]",Background light intensity (0-1)
4,numberOfEpochs,20,Number of epochs
5,onlineAnalysis,extracellular,Online analysis type.
6,orientation,0.0,Center orientation (deg)
7,spatialClass,sinewave,Grating spatial type
8,spatialContrast,0.9,Center grating contrast (0-1)
9,stimTime,60000.0,Stimulus duration (ms)



Parameters that change from epoch to epoch (the condition axes):


,parameter,value,n_levels,comment
0,currentBarWdith,"[50.0, 150.0]",2,
1,currentMeanIntensity,"[0.03, 0.3]",2,


In [4]:
# One row per epoch: the conditions it ran, and what the population did.
epochs = ra.epoch_condition_table(stim_block, response_block,
                                  cell_types = MAIN_TYPES, minimum_n = 3,
                                  source = source)
display(epochs)

# The design as it ran. An unbalanced grid here is worth noticing before it
# becomes an unbalanced comparison later.
print(f'{len(epochs)} epochs over the condition grid:')
display(pd.crosstab(epochs[CONDITION_KEYS[0]], epochs[CONDITION_KEYS[1]]))

,epoch,currentBarWdith,currentMeanIntensity,n_spikes,spikes_per_cell
0,0,50.0,0.03,6079,14.2
1,1,50.0,0.30,17691,41.3
2,2,150.0,0.03,6297,14.7
3,3,150.0,0.30,19870,46.4
4,4,50.0,0.03,18195,42.5
5,5,50.0,0.30,31559,73.7
6,6,150.0,0.03,33221,77.6
7,7,150.0,0.30,80912,189.0
8,8,50.0,0.03,49977,116.8
9,9,50.0,0.30,73251,171.1


20 epochs over the condition grid:


currentMeanIntensity,0.03,0.30
currentBarWdith,,
50.0,5,5
150.0,5,5


## 3. Choose the epochs to analyze

- Set `EPOCH_RANGE` off the `n_spikes` column in §2. That column is the whole input to the decision, so this section is just the decision.
- **Nothing below this cell sees the discarded epochs.** `epochs_kept`, `EPOCH_INDICES` (the same selection as block positions) and `conditions` are built here, and every later section reads those rather than `epochs` or the raw block — so the trim applies once, visibly.

In [5]:
# <-- EDIT ME: the epochs to analyze, read off the n_spikes column in §2.
EPOCH_RANGE = (7, 19)

# The cleaned handles. Everything downstream reads these, not `epochs`.
epochs_kept    = epochs.iloc[slice(*EPOCH_RANGE)].reset_index(drop = True)
EPOCH_INDICES  = epochs_kept['epoch'].astype(int).to_numpy()
conditions     = {key: epochs_kept[key].to_numpy() for key in CONDITION_KEYS}

print(f'analyzing epochs {EPOCH_RANGE[0]}–{EPOCH_RANGE[1] - 1} '
      f'({len(epochs_kept)} of {len(epochs)})\n')

# What the trim left in each cell of the condition grid, and how hard the
# population fired there. Uneven counts here unbalance any comparison across
# conditions made downstream.
display(epochs_kept.groupby(CONDITION_KEYS)['n_spikes']
                   .agg(n_epochs = 'size', median_spikes = 'median')
                   .astype({'median_spikes': int}))

analyzing epochs 7–18 (12 of 20)



n_epochs  median_spikes
currentBarWdith currentMeanIntensity                         
50.0            0.03                         3          76018
                0.30                         3          98746
150.0           0.03                         3          83702
                0.30                         3         126292

## 4. Drop the silent cells

**One criterion, so a rejection has exactly one cause.** A cell is *active* in an epoch when it fires above `MIN_RATE_HZ`, and is kept when it is active in `MIN_ACTIVE_FRACTION` of the epochs it is scored on. Every other `QCThresholds` gate — burstiness, drift, longest silent run — is off. It is a rate rather than a count so the setting transfers between protocols: the same threshold means the same thing on a 60 s epoch and on a 2 s one.

- **Which epochs a cell is scored on is the whole design of the gate.** The conditions alternate, and they are not equivalent tests of whether a cell is alive — one is dim, and a healthy cell answering the stimulus goes quiet there. Score across all of them and the gate rejects cells for doing what the experiment asked: a flat 80%-of-all-epochs rule keeps 20 of 261 cells here, discarding some with median rates of 8 Hz.
- So the gate scores each cell on **the dominant axis at its high-firing level** — the axis whose levels move population rate most (measured, not assumed: mean intensity at 3.5× on this block, where taking whichever axis sorted first would have chosen bar width), at its higher-rate level, within `EPOCH_RANGE` and nowhere else. 90% rather than all of them, so one bad trial is not disqualifying. That level is chosen population-wide, which makes it a property of the block rather than of the cell being judged; the assumption is that no type prefers the *other* level strongly enough to look dead at this one.
- **Then look, twice.** The dropdown renders exactly the epochs the gate scored, for either group of any type — a `dropped` panel should be visibly empty, and if those cells are firing well the threshold is wrong rather than the cells. The mosaic puts the same decision in space: kept filled, dropped open, one panel per type, all on one shared window framed to the cells, because the comparison is between where one type's survivors sit and where another's do (`zoom=False` for the full canvas, when *where on the display* is itself the question).
- **The outcome to check for** is a gate that emptied one part of the array rather than thinning it evenly. This block is the second kind: OnP, OnM and OffM all keep cells left of the ones they drop (OnM medians 31 vs 49 stixels of canvas x), and OffM's 4 survivors of 87 are one clump at the left edge — so anything said about OffM below rests on those 4. Cells that never matched a noise cluster have no RF to draw and are counted in the title.

In [7]:
# Which condition axis actually drives firing rate, and which of its levels is
# the one a live cell should answer? Both measured on the analyzed epochs only.
pop = epochs_kept['n_spikes'].to_numpy()
spread, high_level = {}, {}
for key in CONDITION_KEYS:
    medians = {level: np.median(pop[conditions[key] == level])
               for level in sorted(set(conditions[key]))}
    lo, hi = min(medians.values()), max(medians.values())
    spread[key] = hi / lo if lo > 0 else np.inf
    high_level[key] = max(medians, key = medians.get)

DOMINANT_AXIS  = max(spread, key = spread.get)
DOMINANT_LEVEL = high_level[DOMINANT_AXIS]
for key, ratio in sorted(spread.items(), key = lambda kv: -kv[1]):
    print(f'{key:24s} changes population rate {ratio:5.1f}x across its levels '
          f'(highest at {high_level[key]:g})')

# The epochs the gate scores: analyzed range, dominant axis at its high level.
GATE_EPOCHS = EPOCH_INDICES[conditions[DOMINANT_AXIS] == DOMINANT_LEVEL]

# <-- EDIT ME: the whole criterion.
MIN_RATE_HZ         = 6.0   # a cell is "active" above this ...
MIN_ACTIVE_FRACTION = 0.9   # ... in this fraction of the scored epochs

# Epoch length from the protocol's own timing. Pass this explicitly: the rate
# gate divides by it, and without it every rate metric is NaN and the whole
# block fails QC silently.
block = stim_block.d_epoch_block_params
T_END_MS = sum(float(block.get(k, 0) or 0) for k in ('preTime', 'stimTime', 'tailTime'))

print(f'\n-> scoring on {DOMINANT_AXIS} = {DOMINANT_LEVEL:g}, the level it fires '
      f'hardest at: {len(GATE_EPOCHS)} of the {len(epochs_kept)} analyzed epochs')
print(f'   {", ".join(str(e) for e in GATE_EPOCHS)}')
print(f'   a cell must be active in {MIN_ACTIVE_FRACTION:.0%} of them, where '
      f'active = more than {MIN_RATE_HZ:g} Hz over a '
      f'{T_END_MS / 1000:.0f} s epoch ({MIN_RATE_HZ * T_END_MS / 1000:.0f} spikes)\n')

# One gate on, the rest off, so a rejection has exactly one cause.
thresholds = ra.QCThresholds(
    min_rate_hz                = MIN_RATE_HZ,
    min_frac_epochs_above_rate = MIN_ACTIVE_FRACTION,
    min_frac_non_silent_epochs = None,
    max_cv                     = None,
    max_silent_trial_frac      = None,
    max_silent_run             = None,
    max_drift_score            = None,
    min_reliability_r          = None,
)

qc = ra.block_qc_metrics(response_block, cell_types = MAIN_TYPES,
                         epoch_indices = GATE_EPOCHS, t_end_ms = T_END_MS,
                         min_rate_hz = MIN_RATE_HZ)
qc = ra.filter_cells_by_qc(qc, thresholds)

GOOD_CELLS = qc.query('passes')['cell_id'].astype(int).tolist()
print(f'{len(GOOD_CELLS)} of {len(qc)} cells kept\n')

display(qc.assign(group = np.where(qc['passes'], 'kept', 'dropped'))
          .groupby(['cell_type', 'group'])
          .agg(n = ('cell_id', 'size'),
               median_rate_hz = ('mean_rate_hz', 'median'),
               median_active_frac = ('frac_epochs_above_rate', 'median'))
          .round(2))

currentMeanIntensity     changes population rate   1.3x across its levels (highest at 0.3)
currentBarWdith          changes population rate   1.1x across its levels (highest at 150)

-> scoring on currentMeanIntensity = 0.3, the level it fires hardest at: 6 of the 12 analyzed epochs
   7, 9, 11, 13, 15, 17
   a cell must be active in 90% of them, where active = more than 6 Hz over a 60 s epoch (360 spikes)

39 of 428 cells kept



n  median_rate_hz  median_active_frac
cell_type group                                           
OffM      dropped  216            1.58                 0.0
          kept       6           10.37                 1.0
OffP      dropped  102            2.44                 0.0
OnM       dropped   45            5.38                 0.5
          kept      26           16.43                 1.0
OnP       dropped   26            6.09                 0.5
          kept       7           17.81                 1.0

In [9]:
# Look at both groups, on the epochs the gate actually scored. A 'dropped'
# panel should be visibly empty; if those cells are firing well here, the
# threshold is wrong rather than the cells.
groups = {}
for cell_type in sorted(qc['cell_type'].dropna().unique()):
    rows = qc[qc['cell_type'] == cell_type]
    for label in ('kept', 'dropped'):
        ids = rows[rows['passes'] == (label == 'kept')]['cell_id'].astype(int).tolist()
        if ids:
            groups[f'{cell_type} — {label} ({len(ids)} cells)'] = (cell_type, ids)


def _render(key):
    cell_type, ids = groups[key]
    fig = ra.plot_epoch_rasters(response_block, cell_type, cell_ids = ids,
                                epoch_indices = GATE_EPOCHS,
                                n_first = 3, n_last = 3,
                                title = f'{key}, scored epochs '
                                        f'({DOMINANT_AXIS} = {DOMINANT_LEVEL:g})')
    return None, ra.figure_to_png(fig)


ra.png_browser([(label, label) for label in groups], _render,
               description = 'Show:');

In [ ]:
# The same decision in space: 1.6 sigma RF ellipses, filled where the cell was
# kept and open where it was dropped, every panel on one window framed to the
# cells. Even thinning leaves the survivors tiling the array; a hole means the
# population that remains is a region of retina, not a sample of the one you
# started with. zoom=False for the full canvas instead.
ra.plot_qc_mosaic(pipeline, qc, cell_types = MAIN_TYPES);

## 5. The response in its place: activity on the stimulus

One epoch and one stretch of seconds: each receptive field colored by how hard that cell fired, over a reconstruction of the grating that was on the display at that moment.

- **The two panels are the argument for each other.** A rate map cannot show whether a bright cell fired steadily or emptied a burst into half a second; a raster cannot show whether the cells that responded were the ones the stimulus covered.
- **Pick by `SHOW_ROW`, not by epoch number.** `row` is the position in the kept set and moves whenever `EPOCH_RANGE` changes; `epoch` is the fixed block position, and it is what `grating_geometry`, `grating_frame` and `cell_activity_in_window` take. The cell resolves one to the other and prints both, plus the condition it landed on.
- **Mosaic and stimulus co-register by construction, with no fitted parameter.** The receptive fields were measured in the stimulus's own coordinates: `get_rf_params` returns centers in stixels of the noise grid (already y-flipped for `imshow`), `pixels_per_stixel = canvasSize[0]/numXChecks` = 800/100 scales them to canvas pixels, and `createPresentation` specifies the grating in those same pixels. §7 measures that registration rather than asserting it. The **electrode overlay** is the one thing here that needs a fitted calibration — `show_electrodes=True` maps chip µm onto the canvas, and the chip's rotation and offset are genuine unknowns (`retinanalysis.utils.rig_calibration` fits them). If the two ever disagree, the electrode transform is what is in question.
- **Three ways a frame could look right and be wrong**, all of them handled in `ra.grating_frame`:
  - `barWidths` and `apertureDiameter` are **microns, not pixels**, and the aperture is a **diameter, not a radius** — both `.m` comments say otherwise and the code beside them passes both through `um2pix`. §2 prints those comments verbatim; this is the one place to distrust them.
  - **`um2pix` rounds**: 50 µm at 3.8 µm/pixel is 13 pixels, not 13.16, and the spatial frequency follows the rounded value. Carrying the exact quotient puts the bars visibly out of register by the edge of a 526-pixel aperture.
  - **Outside the aperture is black, not mean** — the mean-colored rectangle is only as large as the aperture and the background is 0, so those cells spent the epoch in darkness. The printout counts them separately for that reason.
- The frame is an **instant** of a drifting pattern (rendered at the window's midpoint), so it shows geometry rather than what any cell integrated. The stimulus panel is scaled to its own range because these means go as low as 0.03; `stim_vmin=0, stim_vmax=1` for absolute luminance.

In [10]:
# What is left to choose from, indexed the way §3 left it. Two numberings are
# live from here on and they are not interchangeable: `row` is the position in
# the kept set, which shifts whenever EPOCH_RANGE changes, while `epoch` is the
# fixed position in the block and is what every block-level call takes. Choose
# by row off this table; the cell resolves it and prints both.
selection = epochs_kept[['epoch', *CONDITION_KEYS, 'n_spikes']].copy()
selection.index.name = 'row'
display(selection)

# <-- EDIT ME: the row of the table above, and the seconds within that epoch.
SHOW_ROW = 0
WINDOW_S = (10.0, 12.0)

if not 0 <= SHOW_ROW < len(epochs_kept):
    raise IndexError(f'SHOW_ROW must be 0–{len(epochs_kept) - 1} '
                     f'(the rows above); got {SHOW_ROW}')
SHOW_EPOCH  = int(epochs_kept.loc[SHOW_ROW, 'epoch'])
SHOW_LABEL  = ra.condition_label(CONDITION_KEYS, epochs.loc[SHOW_EPOCH])

# Everything about the stimulus comes from this epoch's own recorded
# parameters, so the alternating conditions come out right without being
# looked up anywhere.
geom = ra.grating_geometry(stim_block, SHOW_EPOCH)

# The frame is an instant of a drifting pattern — take the middle of the
# window, so it is representative of what the rates beside it were counted on.
STIM_FRAME, _ = ra.grating_frame(stim_block, SHOW_EPOCH,
                                 time_s = float(np.mean(WINDOW_S)))

# The population as it will be drawn: the §4 survivors, each with its rate in
# the window and its receptive field placed on the canvas.
activity = ra.cell_activity_in_window(pipeline, SHOW_EPOCH, WINDOW_S,
                                      cell_types = MAIN_TYPES,
                                      cell_ids   = GOOD_CELLS)

mpp       = geom['microns_per_pixel']
window    = WINDOW_S[1] - WINDOW_S[0]
period_um = 2 * geom['bar_width_px'] * mpp

print(f'row {SHOW_ROW} of {len(epochs_kept)} kept -> block epoch {SHOW_EPOCH}, '
      f'{WINDOW_S[0]:g}–{WINDOW_S[1]:g} s  ({SHOW_LABEL})\n')
print(f"  mean intensity   {geom['mean_intensity']:g} at contrast {geom['contrast']:g}, "
      f"so the frame runs {geom['mean_intensity'] * (1 - geom['contrast']):.3f}"
      f"–{geom['mean_intensity'] * (1 + geom['contrast']):.3f} inside the aperture "
      f"and 0 outside it")
print(f"  bar width        {geom['bar_width_um']:g} µm -> {geom['bar_width_px']} pix "
      f"(um2pix rounds), a {2 * geom['bar_width_px']} pix = {period_um:.0f} µm period")
print(f"  aperture         {geom['aperture_diameter_um']:g} µm "
      f"-> {geom['aperture_diameter_px']} pix across, centred on the "
      f"{geom['canvas_w']}×{geom['canvas_h']} canvas")
print(f"  drift            {geom['temporal_freq_hz']:g} Hz at orientation "
      f"{geom['orientation_deg']:g}°, so {geom['temporal_freq_hz'] * window:g} cycles "
      f"pass in this window")

# How the bars compare to a receptive field decides whether the grating is
# resolvable at all: bars much finer than an RF largely cancel within it.
rf_um = np.nanmedian(activity['width']) * mpp
print(f"\n  the RF ellipses as drawn are a median {rf_um:.0f} µm across, against "
      f"{period_um:.0f} µm of grating period")

# Cells outside the aperture saw darkness, not the grating. Their rate is a
# real number about a different stimulus, so read the two groups apart.
r_px    = np.hypot(activity['center_x'] - geom['center_x'],
                   activity['center_y'] - geom['center_y'])
outside = r_px > geom['aperture_diameter_px'] / 2
print(f"\n  {(~outside).sum()} of {len(activity)} cells have their RF centre inside the "
      f"aperture (median {activity.loc[~outside, 'rate_hz'].median():.2f} Hz)")
print(f"  {outside.sum()} sit outside it, in the black surround "
      f"(median {activity.loc[outside, 'rate_hz'].median():.2f} Hz) — whatever those "
      f"cells are doing, it is not a response to the grating")

,epoch,currentBarWdith,currentMeanIntensity,n_spikes
row,,,,
0,7,150.0,0.30,114439
1,8,50.0,0.03,20176
2,9,50.0,0.30,80745
3,10,150.0,0.03,28208
4,11,150.0,0.30,132538
5,12,50.0,0.03,32153
6,13,50.0,0.30,90061
7,14,150.0,0.03,34122
8,15,150.0,0.30,154578


row 0 of 13 kept -> block epoch 7, 10–12 s  (currentBarWdith 150, currentMeanIntensity 0.3)

  mean intensity   0.3 at contrast 0.9, so the frame runs 0.030–0.570 inside the aperture and 0 outside it
  bar width        150 µm -> 39 pix (um2pix rounds), a 78 pix = 296 µm period
  aperture         2000 µm -> 526 pix across, centred on the 800×600 canvas
  drift            2 Hz at orientation 0°, so 4 cycles pass in this window

  the RF ellipses as drawn are a median 186 µm across, against 296 µm of grating period

  48 of 55 cells have their RF centre inside the aperture (median 33.00 Hz)
  7 sit outside it, in the black surround (median 23.00 Hz) — whatever those cells are doing, it is not a response to the grating


In [11]:
# Left: the mosaic over the stimulus, each RF filled by its rate in the window
# and outlined in its cell type's colour, with the aperture marked. Right: the
# spikes those rates were counted from, sorted within each type by rate so the
# fill gradient and the density gradient run the same way.
#
# All four types at once is a crowded picture — 1.6 sigma ellipses overlap
# heavily at this density — so the dropdown also takes one type at a time,
# which is the view to use for anything but a first look.
_types = [ct for ct in MAIN_TYPES if (activity['cell_type'] == ct).any()]


def _render_activity(key):
    fig = ra.plot_mosaic_activity(
        pipeline, SHOW_EPOCH, WINDOW_S,
        stim_frame           = STIM_FRAME,
        cell_types           = _types if key == 'all types' else [key],
        cell_ids             = GOOD_CELLS,
        aperture_diameter_px = geom['aperture_diameter_px'],
        title                = f'epoch {SHOW_EPOCH} — {SHOW_LABEL} — '
                               f'{WINDOW_S[0]:g}–{WINDOW_S[1]:g} s',
    )
    return None, ra.figure_to_png(fig)


ra.png_browser([(k, k) for k in ['all types'] + _types], _render_activity,
               description = 'Cell type:');

### 5b. The same thing moving: a clip of the response over the drifting grating

The figures above are stills of something that moves. They answer the spatial question — did the cells the grating covered respond — and leave the temporal one alone. But a drifting grating drives cells half a spatial period apart in antiphase, so the thing actually worth seeing is the response **travelling across the mosaic in step with the bars**, and no still shows that. `ra.animate_mosaic_activity` renders it: same two panels, same coordinates, one axis more.

**Three clocks have to agree, and only one of them is the spike clock.**

- Spike times are milliseconds from the **epoch start** — that is the clock the whole package uses.
- `grating_frame(time_s=...)` wants seconds from **stimulus onset**, which is the epoch start plus `preTime`. They coincide only when `preTime` is zero, which it is for this block and is not in general. Pass `pre_time_ms = geom['pre_time_ms']` and the conversion happens in one place instead of being silently assumed.
- **The retina answers late.** A firing rate at time *t* reports a stimulus from some tens of milliseconds earlier, so the response trails the bars on screen. That lag is a measurement, not an error, and `latency_ms = 0` leaves it in. Set `latency_ms` to shift the rate sampling forward and lock the two together — but do that *after* looking at the uncorrected clip, because building the alignment in by default would destroy the evidence for it.

**What the lag is here.** Correlating each cell's rate against the luminance the grating put over its own receptive field, on this epoch, the OnM population peaks at **r = 0.63 at +35 ms** — a real latency, and the movie's validity check, since nothing in that prediction is fitted. OnP is weakly driven (6 cells survive §4) and OffP/OffM sit at r ≈ 0.03, which is the same answer §7 reaches by a different route. One caveat: a 2 Hz drive is periodic at 500 ms, so a lag measured this way is only identified modulo 500 ms — +35 ms is the physiological reading of it, not the only arithmetic one.

**Two things are held fixed across every frame**, because an animation that rescales per frame encodes nothing: the rate colour scale (one colour means one firing rate from the first frame to the last) and the stimulus display range (otherwise the background pulses on its own). The rate is a Gaussian-smoothed spike train, and `rate_sigma_s` is the whole temporal resolution of the clip — it has to be short against the 500 ms drift period, so 30 ms resolves the modulation where 150 ms would average it flat.

`speed` is playback rate, not a change of what is shown: `0.25` renders four times the frames and plays back in slow motion, which a 2 Hz drift needs to be followable by eye. Write `.mp4` (OpenCV) or `.gif` (pillow) — the extension picks the writer.

In [ ]:
import os
from IPython.display import Video, Image as IPyImage

# <-- EDIT ME: the seconds to animate, and how to play them back.
MOVIE_WINDOW_S   = (10.0, 11.5)   # 3 drift cycles at 2 Hz; keep it short
MOVIE_SPEED      = 0.25           # 0.25x, i.e. four times the frames, slow-mo
MOVIE_FPS        = 20
MOVIE_SIGMA_S    = 0.03           # rate kernel; short against the 500 ms period
MOVIE_LATENCY_MS = 0.0            # 0 leaves the retina's lag visible; try 35

MOVIE_DIR  = os.path.join(ra.OUTPUT_DIR, EXP_NAME, 'variableMeanDriftingGrating')
os.makedirs(MOVIE_DIR, exist_ok = True)
MOVIE_PATH = os.path.join(
    MOVIE_DIR, f'epoch{SHOW_EPOCH}_activity_{MOVIE_WINDOW_S[0]:g}-'
               f'{MOVIE_WINDOW_S[1]:g}s.mp4')

# The stimulus is supplied rather than rendered inside, the same way §5 passes
# STIM_FRAME — what a frame *is* differs per protocol. This one is called with
# seconds from stimulus onset, which is why pre_time_ms goes in beside it.
# downsample=2 renders the grating at 400x300; it is a background, not the
# subject, and full canvas resolution only costs frames.
def _grating_at(t_from_onset_s):
    frame, _ = ra.grating_frame(stim_block, SHOW_EPOCH,
                                time_s = t_from_onset_s,
                                geometry = geom, downsample = 2)
    return frame


ra.animate_mosaic_activity(
    pipeline, SHOW_EPOCH, MOVIE_WINDOW_S, MOVIE_PATH,
    stim_frame_fn        = _grating_at,
    pre_time_ms          = geom['pre_time_ms'],
    cell_types           = MAIN_TYPES,
    cell_ids             = GOOD_CELLS,
    aperture_diameter_px = geom['aperture_diameter_px'],
    fps                  = MOVIE_FPS,
    speed                = MOVIE_SPEED,
    rate_sigma_s         = MOVIE_SIGMA_S,
    latency_ms           = MOVIE_LATENCY_MS,
    title                = f'epoch {SHOW_EPOCH} — {SHOW_LABEL}',
)

# Embedded so the clip travels with the notebook rather than pointing at a
# path on this machine.
Video(MOVIE_PATH, embed = True) if MOVIE_PATH.endswith('.mp4') else IPyImage(MOVIE_PATH)

## 6. Is it one cell? Sorting QC on the raw trace

Rasters and PSTHs test whether spike *times* follow the stimulus. They cannot test whether the spikes were assigned to the right cell: a merge of two units and a genuinely well-driven cell both produce a tidy, stimulus-locked raster. Only the raw voltage separates them.

- **A few cells per type, sampled at random rather than by rate.** The highest-firing cells are the large, well-isolated ones that are easiest to sort, so a top-rate sample inspects the cells least likely to be wrong. `SORT_SEED` makes a draw recoverable; `None` for a fresh one. The pool is §4's table in memory — `ra.sample_cells_by_type` takes `qc.query('passes')` directly, so this works in a notebook that never archives a `qc.csv`.
- **One epoch, about a second.** `ra.load_raw_window` keeps all electrodes but only those samples: ~15 MB against ~920 MB for the full epoch, which matters because raw `.bin` is canonical on the NAS. Take the second literally — a spike is about a millisecond wide, so five seconds in one panel is 100 000 samples across ~1300 pixels and every waveform collapses to a single pixel. The panel still renders, and an unreadable one looks like a sorting problem rather than a plotting one.
- **The trace is high-passed at 300 Hz**, which is what the sorter saw. Unfiltered, the drift on these electrodes is tens of counts — the same size as a spike — so it slides the baseline under the marks and buries the waveforms in a band. The filter is zero-phase for a reason: a causal one would delay the trough away from the spike sample and manufacture exactly the misalignment this section exists to detect. `hp_cutoff_hz=None` shows the unfiltered trace.
- **Reading it**: the trace is the cell's strongest electrode, red marks are its spikes, other colors are other cells on that same electrode. A clean sort puts every red mark on a visible downward deflection. Clear waveforms with *no* red mark mean spikes are missing or went to a neighbor; red marks on flat baseline mean template hits that are not spikes. Several same-electrode neighbors firing in near-lockstep is the split-cluster signature — `ra.dedup_pipeline` is the tool for it.
- **If it says the raw is unreachable while the NAS is mounted**, the mount happened after `import retinanalysis`. Volumes are discovered once, at import, and a section of `config.ini` whose root did not exist then is dropped from the tier list for the life of the kernel — so the NAS is not merely unsearched, it is unknown. `ra.reload_config()` rediscovers the mounts in place, which beats restarting a kernel holding a built pipeline. The error names the tiers it skipped and why.

In [ ]:
# <-- EDIT ME: how many cells to check, and which epoch and seconds to check them on.
SORT_TYPES       = MAIN_TYPES
N_CELLS_PER_TYPE = 2
SORT_EPOCH       = SHOW_EPOCH      # the epoch §5 drew; any block epoch works
SORT_WINDOW_S    = (10.0, 11.0)    # ~1 s is what a raw trace can actually show
SORT_SEED        = 0               # None for a fresh draw each run

# Sampled from §4's table in memory — random, not top-rate, so the check lands
# on ordinary cells rather than the easiest-to-sort ones.
sort_sample = ra.sample_cells_by_type(qc.query('passes'),
                                      cell_types       = SORT_TYPES,
                                      n_cells_per_type = N_CELLS_PER_TYPE,
                                      random_seed      = SORT_SEED)
display(sort_sample)

# One windowed read of the raw .bin, shared by every cell below. Returns None
# with a printed reason when the raw store is unreachable rather than raising,
# since this is the only section that needs it. If that reason is a tier that
# was not mounted at import — the NAS, usually — run ra.reload_config() and
# re-run this cell; the pipeline in the kernel survives it.
raw = ra.load_raw_window(response_block, SORT_EPOCH, SORT_WINDOW_S)

# Only the §4 survivors count as same-electrode competition; the unmatched
# clusters would bury the panel in legend. The trace is high-passed at 300 Hz
# so the marks can be judged against the waveforms (hp_cutoff_hz=None to see
# what is actually on the electrode).
ra.browse_sorting_qc(raw, response_block, sort_sample, SORT_EPOCH,
                     candidate_cell_ids = GOOD_CELLS);

## 7. Phase alignment: does the response follow the grating in space?

A drifting grating gives every point on the display its own temporal phase — luminance there is `mean · (1 + contrast · sin(2π f a + 2π F t))`, so cells half a spatial period apart are driven in antiphase. A cell's F1 phase is therefore **predicted by where its receptive field sits**: one cycle of phase per spatial period, along the drift axis and nowhere else. Nothing in that prediction is fitted, which makes it useful twice.

- **As a measurement of the registration.** §5 argued that the mosaic and the stimulus share a frame by construction; "by construction" is an argument, not a measurement. Scanning candidate periods × orientations for the one that makes the residual phases agree recovers the grating from spike times and STA centers alone. Here, at 150 µm bars and mean 0.3, the phases pick **78.5 px at 0°** against a stimulus of **78 px at 0°** — 1% in period and a couple of degrees in orientation, which is also the bound on any spatial offset between the two frames.
- **As the resolvability question.** Bars finer than a receptive field cancel within it, and then there is no F1 to have a phase. That is what 50 µm and 150 µm in one block are asking, and only the coarse bars at the high mean are resolved: at 50 µm the period is 99 µm against a median RF of ~186 µm, and the scan picks 20 px against a true 26 with a concentration below its own null.
- **Read every peak against that null.** With few driven cells a grid search finds a peak in noise — the dim conditions here reach R = 0.78–0.87 on 13–16 cells, of which 1–7 are modulated at p < 0.01. `n_shuffles` permutes the receptive-field positions among the cells and rescans, reporting the same max-over-grid statistic, so the null includes whatever the search itself buys. Two of the four conditions come out nominally "above" their null, but 50 µm at mean 0.03 clears it by 0.000 on 13 cells — a tie is not an alignment, and the margin is what to read rather than the verdict.
- **The residual** — `resp_phase − stim_phase` — is what holds constant when the prediction holds, and what it holds at is the response latency expressed as a phase, so a latency mod one drift cycle (500 ms at 2 Hz). ON and OFF cells sit half a cycle apart, which is why concentration is computed per cell type and why types with fewer than 3 cells are left out: a lone cell agrees with itself at every candidate period.

Reading the figure — **left**: response phase against position across the bars, with the predicted sawtooth drawn through each type's own mean residual, so what is judged is the slope and not an offset that latency and polarity are free to set. **Middle**: the residual on the mosaic relative to each type's mean — flat color is alignment, a left-to-right gradient means the period is off, a patch of unrelated color means those cells are not following the grating. **Right**: concentration against candidate period, with the stimulus period and the shuffled null marked.

### The drift frequency is not 2 Hz

Everything above folds spikes at the temporal frequency, so it is worth establishing that the frequency is the one the retina saw. It is not the one the protocol declares.

**Stage advances the grating one phase increment per rendered frame, sized from the *declared* refresh rate, and the display then runs at its own.** The recorded frame times for this block give **60.31 Hz** against a declared 60, so a nominal 2 Hz grating drifts at 2 × 60.31/60 ≈ **2.01 Hz**. `ra.estimate_drift_frequency` recovers **2.0099 Hz** from the spikes alone — implying 60.30 Hz, agreeing with the frame times to 0.03 %.

Over a 60 s epoch that fraction of a percent is not small:

- Pooled vector strength is **0.285** at the nominal 2.0000 Hz and **0.506** at 2.0099. Folding at the declared value discards **44 %** of the measured modulation, and here it halves the median F1 of the resolvable condition (0.58 → 0.29).
- The phase error accumulates to **215°** from the start of an epoch to the end, so a residual averaged over the whole epoch is an average over a phase that has drifted most of a cycle. It is not an estimate of anything.
- **The latency is the tell.** Folded at the nominal frequency the mean residual for OnM is −47°, a *negative* lag of −66 ms: the response preceding the stimulus. Corrected, it is +49° → **+68 ms** (OnP +84 ms) — positive, physiological, and the same sign and order as the +35 ms §5b measured by cross-correlating rate against local luminance, which is an independent route through different machinery.
- It also fixes the **dim coarse** condition: at 150 µm and mean 0.03 the scan picked 52 px against a true 78, and now picks **79.4 px**. Its concentration still fails to clear its own null (0.780 against 0.785), so this is a better estimate of something still not significant, not a new result.

What the correction does **not** touch is the period and orientation the scan picks for the resolvable condition — 77.5 px at 0°, either way. That is a spatial fit at fixed temporal frequency, so a frequency error moves every cell's phase together and leaves the slope across position alone. The registration claim in the first bullet stands unchanged; the strengths and the latencies did not.

`DRIFT_FREQ_HZ` is estimated in the cell below and passed to §8 and §9 as well, since every condition ran on the same display.

In [ ]:
# One scan per condition, because a phase estimate may only pool epochs that
# ran the same geometry: drift phase is measured from stimulus onset, so
# epochs of one condition are in register and their F1 vectors add.
SCAN_SHUFFLES = 100     # position-shuffle null the peak has to beat; 0 to skip

# The frequency the display actually delivered, recovered from the spikes.
# Estimated on the best-driven condition — one display, so one estimate serves
# every condition — and reused by §8 and §9. Set DRIFT_FREQ_HZ = None to fold
# at the declared value and watch the latency go negative.
_best = (epochs_kept.groupby(CONDITION_KEYS)['n_spikes'].sum().idxmax())
_best_epochs = epochs_kept.loc[
    (epochs_kept[CONDITION_KEYS[0]] == _best[0])
    & (epochs_kept[CONDITION_KEYS[1]] == _best[1]), 'epoch'].astype(int).tolist()
DRIFT = ra.estimate_drift_frequency(
    pipeline, stim_block, _best_epochs,
    geometry   = ra.grating_geometry(stim_block, _best_epochs[0]),
    cell_types = MAIN_TYPES)
DRIFT_FREQ_HZ = DRIFT['drift_freq_hz']
print(f'  estimated on {ra.condition_label(CONDITION_KEYS, _best)}, '
      f'epochs {_best_epochs}\n')

# Per condition and per cell: F1 strength and phase, the phase the grating
# predicts at that cell's position, and the period/orientation that best make
# those phases agree. Each epoch's geometry comes from its own parameters.
phase_by_condition, summary = ra.phase_alignment_by_condition(
    pipeline, stim_block, epochs_kept, CONDITION_KEYS,
    cell_types = MAIN_TYPES,
    cell_ids   = GOOD_CELLS,
    n_shuffles = SCAN_SHUFFLES,
    drift_freq_hz = DRIFT_FREQ_HZ)
display(summary.round(3))

# Which conditions beat their own shuffled null — the only ones that are
# evidence of anything — and, for the best of them, the mean residual per cell
# type read as a latency (mod one drift cycle).
alignment = ra.describe_phase_alignment(summary, phase_by_condition,
                                        CONDITION_KEYS)

In [ ]:
# The same conditions as figures. Each label carries that condition's
# concentration against its null, so the one that aligned is findable without
# rendering the rest — and the ones that did not are worth a look too, since a
# failed alignment and a resolved-but-offset one look nothing alike.
ra.browse_phase_alignment(phase_by_condition, CONDITION_KEYS);

## 8. The response folded on the drift cycle

Before measuring anything about the population, look at what the response *is*. Three steps, each needing the one before it, and the middle one is where this protocol hides a trap.

1. **Trial-averaged PSTH per cell** (`ra.cell_mean_psth`, `ra.browse_cell_psths`). One trace per cell, averaged over the epochs of one condition. The slow adaptation shows up here as a decaying envelope, and a cell that is not driven at all announces itself.
2. **Cycle average per cell** (`ra.cycle_average`). The drift frequency is known, so the epoch folds onto one cycle and the modulation can be read instead of inferred from an F1 amplitude. Folding ~120 cycles into one is also what makes a 10 Hz cell's modulation visible at all.
3. **Type average, aligned by position** (`ra.aligned_cycle_average`). **This step cannot be skipped.** Cells half a spatial period apart are driven in antiphase, so averaging their folds directly cancels them: for OnM here the type mean of unaligned folds keeps **0.10** of the single-cell modulation, and after rotating each cell by the phase the grating puts at its own receptive field it keeps **0.90**. Averaged the naive way, the population looks unmodulated precisely when it is best organised.

   The rotation is `pi/2 - 2*pi*f_s*a` at position `a` across the bars — §7's prediction, nothing fitted. `ra.plot_cycle_alignment` draws cells against phase before and after: **the diagonal in the left panel is the spatial code** (one cycle of response phase per spatial period), and **that it stands up vertical in the middle panel is the check** that the rotation is right rather than merely applied. Rows are blocked by cell type, because ON and OFF are antiphase and sorting the whole population by position alone cross-hatches the two diagonals into noise.

### Folded at the corrected frequency

§7 established that the display delivers 2.0099 Hz rather than the declared 2, and `DRIFT_FREQ_HZ` is passed to every fold below. It matters most here: folded at the nominal value the aligned OnM response marches **316° to 150°** across the epoch, which reads as a latency growing by a quarter of a second with adaptation. Corrected, the phase is **flat at 300–307°** while modulation depth grows 0.38 → 0.62. The drift was the artefact; the depth is the signal, and the two are only separable once the frequency is right.

**Below the heat map, the same rows as tuning curves.** Colour is hard to read quantitatively and the 98th-percentile scale hides whatever sits above it, so `plot_cycle_evolution` draws a few time snippets — early dark, late bright — as phase tuning curves with an amplitude axis under them. A coloured tick at the left edge of the heat map marks which row each curve came from. `cross_sections=[...]` picks the windows by hand, `n_cross_sections` changes how many the automatic choice takes.

The curves share a y-axis across types so amplitudes compare, which is what makes OffP worth a glance: three cells, dividing by a near-zero mean, reaching four times OnM's modulation and squashing everything else onto a sliver. That is the honest picture of a thinly sampled type, and `share_curve_y=False` gives each type its own scale once you have seen it.

In [ ]:
# Epochs of one condition — the same rule as everywhere else, since epochs of
# a different bar width are a different stimulus and a different mean is a
# different adaptation state.
CYCLE_COND   = (150.0, 0.30)
CYCLE_LABEL  = ra.condition_label(CONDITION_KEYS, CYCLE_COND)
CYCLE_EPOCHS = epochs_kept.loc[
    (epochs_kept[CONDITION_KEYS[0]] == CYCLE_COND[0])
    & (epochs_kept[CONDITION_KEYS[1]] == CYCLE_COND[1]), 'epoch'
].astype(int).tolist()
CYCLE_GEOM   = ra.grating_geometry(stim_block, CYCLE_EPOCHS[0])
print(f'{CYCLE_LABEL}: epochs {CYCLE_EPOCHS}\n')

# DRIFT_FREQ_HZ comes from §7 — one display, one estimate.
print(f'folding at {DRIFT_FREQ_HZ:.4f} Hz\n')

# Step 1: one trial-averaged trace per cell, ordered across the bars so
# neighbouring panels are neighbouring retina.
cells_psth, t_psth, psth = ra.cell_mean_psth(
    pipeline, CYCLE_EPOCHS, cell_types = MAIN_TYPES, geometry = CYCLE_GEOM)
print(f'\n{len(cells_psth)} cells, PSTH over {t_psth[-1]:.0f} s')
ra.browse_cell_psths(cells_psth, t_psth, psth,
                     cell_types = MAIN_TYPES, geometry = CYCLE_GEOM);

In [ ]:
# Steps 2 and 3. One fold per cell over the whole epoch, then the same folds
# rotated by the phase each cell's own position predicts.
N_CYCLE_BINS = 24     # finer than §9's 12: the rotation is to the nearest bin

pbr_cycle = ra.phase_binned_response(
    pipeline, stim_block, CYCLE_EPOCHS,
    windows_s     = [(0.0, 60.0)],
    n_phase_bins  = N_CYCLE_BINS,
    cell_types    = MAIN_TYPES,
    drift_freq_hz = DRIFT_FREQ_HZ)

# Left panel diagonal = the spatial code; middle panel vertical = the rotation
# is right. The number beside each type in the legend is the fraction of
# single-cell modulation its mean retains, which is the cost of skipping this.
fig = ra.plot_cycle_alignment(pbr_cycle, cell_types = MAIN_TYPES)
plt.show()

# Step 4: the same thing in 5 s windows, so the fold becomes an image of phase
# against time since the step. normalize='fraction' is the default and the
# only setting that shows the recovery — in Hz the modulation shrinks, because
# rate falls faster than F1/F0 rises, and z-scored it is flat by construction.
CYCLE_WINDOWS = ra.sliding_windows(60.0, width_s = 5.0, step_s = 5.0)
pbr_time = ra.phase_binned_response(
    pipeline, stim_block, CYCLE_EPOCHS,
    windows_s     = CYCLE_WINDOWS,
    n_phase_bins  = N_CYCLE_BINS,
    cell_types    = MAIN_TYPES,
    drift_freq_hz = DRIFT_FREQ_HZ,
    verbose       = False)

evolution = ra.cycle_evolution(pbr_time, cell_types = MAIN_TYPES)

# The depth grows and the phase holds still. Run this with
# drift_freq_hz=None above and the phase column marches instead — that is the
# artefact the section header is about, not a latency.
w = np.exp(-1j * 2 * np.pi * np.arange(N_CYCLE_BINS) / N_CYCLE_BINS)
print(f'{"type":6} {"depth over the epoch":>44}   {"aligned phase (deg)":>26}')
for ct in MAIN_TYPES:
    if ct not in evolution:
        continue
    z = np.asarray(evolution[ct]) @ w
    depth, ph = np.abs(z) / N_CYCLE_BINS, np.degrees(np.angle(z)) % 360
    print(f'{ct:6} {depth[0]:.2f} -> {depth[-1]:.2f}   '
          f'[{" ".join(f"{v:.2f}" for v in depth)}]   '
          f'{ph[0]:3.0f} -> {ph[-1]:3.0f}')

fig = ra.plot_cycle_evolution(
    evolution, title = f'{CYCLE_LABEL} — 5 s windows, folded at '
                       f'{DRIFT_FREQ_HZ:.4f} Hz')
plt.show()

## 9. Recovery of spatial sensitivity after the luminance step

Each epoch begins with a luminance step, so the analysis asks how the spatial code changes with time since that step. Conditions stay separate because bar width and mean define different stimuli. The computation is now one reusable call per condition; its output is converted immediately to a tidy table that is also the cross-date storage contract.

The primary endpoints are bias-corrected F1/F0, spike-count-matched phase decoding, polarity-blind decoding, coherence, and reliability. Cell selection is fixed over the whole epoch, vector-strength bias is corrected, and matched decoding thins every window to the same spike density. For pooling dates, rate and F1 are normalized to each retina’s late state, while decoding is normalized from that retina’s own chance/shuffle level to its late state. Dates—not cells—are the biological replicates.

The focus condition remains the coarse grating at high mean (`150 µm`, `0.30`), the condition whose spatial alignment is resolvable in this block. Fine-grating conditions are retained in the saved table, but their normalized decoding index is left undefined when the late response does not clear its null.

In [ ]:
# Shared settings for every condition.
STEP_WINDOWS = ra.recovery_windows([0, 2, 5, 10, 20, 30, 45, 60])
N_PHASE_BINS = 12
N_SHUFFLES   = 50

# One high-level call replaces the repeated modulation/decoder/coherence loop.
recovery = ra.analyze_recovery_conditions(
    pipeline, stim_block, epochs_kept,
    condition_keys = CONDITION_KEYS,
    windows_s      = STEP_WINDOWS,
    cell_types     = MAIN_TYPES,
    drift_freq_hz  = DRIFT_FREQ_HZ,
    n_phase_bins   = N_PHASE_BINS,
    n_shuffles     = N_SHUFFLES,
)

# One row per condition × window, including raw and within-date-normalized
# endpoints. This table is the object saved in §11.
recovery_summary = ra.recovery_summary_table(recovery, exp_name = EXP_NAME)
display(recovery_summary.set_index(['condition', 'window'])[[
    'rate_hz', 'f1', 'f1_naive', 'f2_over_f1', 'n_f1_resolved',
    'decode_matched', 'decode_matched_index', 'decode_polblind',
    'decode_polblind_index', 'reliability',
]].round(3))

In [ ]:
# The condition to characterize and later pool across dates.
FOCUS = ra.condition_label(CONDITION_KEYS, (150.0, 0.30))
r = recovery[FOCUS]
focus_summary = recovery_summary.query('condition == @FOCUS')

print(f'{EXP_NAME}: {FOCUS}\n')
display(focus_summary.set_index('window')[[
    'rate_late_fraction', 'f1_late_fraction',
    'decode_matched_index', 'decode_polblind_index',
]].round(3))

# A single-retina timescale is descriptive; dates become the bootstrap unit
# only after multiple preparations have been saved.
mod_t = r['modulation'].groupby('t_mid')['m1'].mean()
fit = ra.fit_recovery(mod_t.index.to_numpy(), mod_t.to_numpy())
print(f'F1 recovery: tau={fit["tau_s"]:.1f} s, '
      f't50={fit["t50_s"]:.1f} s, R²={fit["r_squared"]:.3f}')
if not fit['tau_bounded']:
    print('tau is not bounded above within this epoch; do not quote it')

RECOVERY_FIG = ra.plot_recovery_summary(
    r['modulation'], r['full'], r['polarity_blind'],
    r['coherence'], r['reliability'],
    title = f'recovery after the step — {FOCUS}',
)
plt.show()

## 10. Where to go from here

**The cleaned handles**: `epochs_kept` (analyzed epochs and their conditions), `EPOCH_RANGE` / `EPOCH_INDICES` (the same selection as a slice and as block positions), `conditions` (per-axis levels, one entry per kept epoch), `GOOD_CELLS`, `phase_by_condition` (per condition: the phase table, its scan, its geometry) and `alignment` (which conditions beat their null, and the per-type latency of the best one). Write downstream code against those, never against `epochs` or the raw block. Anything indexing the block — `get_spike_xarr`, a raster, a PSTH — takes `EPOCH_INDICES`; anything reasoning about conditions lines up row-for-row with `conditions`.

Next steps this notebook sets up:

- **The response across the grid.** PSTHs per (mean intensity × bar width), and how mean intensity shifts the grating response at each bar width. `ra.get_spike_xarr(response_block, cell_types=MAIN_TYPES)` gives the ragged (cell × epoch) array; slice with `EPOCH_INDICES`, select with `GOOD_CELLS`.
- **§5 as a measurement rather than a look.** `ra.cell_activity_in_window` returns rates as a table, so calling it across `EPOCH_INDICES` and grouping by `conditions` turns one snapshot into a condition comparison — split by whether the grating period was above or below each cell's RF diameter, which is the quantity that decides resolvability.
- **§7 per cell rather than per population.** `drift_phase_response` already returns per-cell F1 strength and phase; comparing strength between bar widths for the same cell is a contrast-sensitivity-by-spatial-frequency measurement on 89 cells, and the residual is a per-cell latency.

Four things to carry forward:

- `GOOD_CELLS` was chosen on the high-firing level of `DOMINANT_AXIS` alone. Right for asking "is this cell alive", but not neutral with respect to that axis — a type's response *at the low level* is measured on cells picked for firing at the high one. Fine for describing how a response changes across the axis; not a basis for what fraction of cells respond at the low level.
- The tenfold rise in population rate across this block is monotonic and present in both conditions. Comparisons *between* conditions are safe (they alternate, so both sample the trend evenly); comparisons between early and late epochs are confounded with it.
- The aperture is 2000 µm and the array is wider, so a standing fraction of the population never saw the grating — 27 of 116 cells here. §5 counts them per epoch and §7 excludes them; they belong in a surround analysis, not in an average labelled as the response to the stimulus.
- **Only one of the four conditions passes §7's alignment test.** Anything claiming a spatial response to the grating rests on 150 µm bars at mean 0.3; at 50 µm the population is not resolving the bars, so a rate difference there is a response to the mean, not to the pattern.

Before trusting any of it, §6 is the check the rest of the notebook cannot make: a merge and a well-driven cell produce the same tidy raster, and only the raw trace separates them.

## 11. Save this date and update the cross-date dataset

`ra.save_recovery_summary` first prints every date already saved, then writes the current experiment’s pickle, readable JSON metadata, and single-date plot under `<OUTPUT_DIR>/protocol_analysis/vmdg/<date>/`. Re-running a date replaces that date; adding a date leaves earlier experiments untouched. The combined loader reads all date bundles into one tidy dataset.

Normalization happens within each date and condition before pooling: rate and F1 are fractions of that retina’s late value; decoder indices run from its own chance/shuffle level (`0`) to its late state (`1`). The pooled mean therefore weights each retina once rather than weighting dates by their number of cells. The combined pickle, JSON metadata, and multi-date figure are saved under `<OUTPUT_DIR>/protocol_analysis/vmdg/summary/`.

In [ ]:
# This function prints saved-date coverage before writing the current date.
date_bundle = ra.save_recovery_summary(
    recovery_summary, EXP_NAME,
    metadata = {
        'datafile_name': DATAFILE_NAME,
        'analysis_chunk': ANALYSIS_CHUNK,
        'focus_condition': FOCUS,
        'step_windows_s': STEP_WINDOWS,
        'n_phase_bins': N_PHASE_BINS,
        'drift_freq_hz': DRIFT_FREQ_HZ,
    },
    figures = {'recovery_summary': RECOVERY_FIG},
)

# Reload from disk so this object represents the complete accumulated dataset,
# including the date just saved.
ALL_RECOVERY = ra.load_recovery_many()
print(f'\nCombined dataset: {ALL_RECOVERY["exp_name"].nunique()} dates, '
      f'{len(ALL_RECOVERY)} condition-window rows')
display(ra.saved_recovery_stats())

### Cross-date recovery plot

Gray trajectories are individual retinas. The black line is the date-level mean and the band is SEM across dates. This is the pooled analysis view to rerun after each new experiment is saved.

In [ ]:
POOLED_CONDITION = FOCUS
POOLED_FIG, axes = ra.plot_recovery_across_dates(
    ALL_RECOVERY, condition = POOLED_CONDITION)
summary_bundle = ra.save_recovery_cross_date_summary(
    ALL_RECOVERY,
    metadata = {'focus_condition': POOLED_CONDITION},
    figures = {'cross_date_recovery': POOLED_FIG},
)
plt.show()